# 01 — What θ actually is, and what msprime produces

Goal: see the parameter vector θ and the simulated genotype matrix as **concrete Python objects**, not abstractions.

We use the paper's actual 9-parameter `dadi_joint_mig` model throughout — same model behind Table 1.

Each cell touches one concept we've discussed:
- Cell 1: setup
- Cell 2: θ as a `ParamSet` object — all 9 demographic params
- Cell 3: θ update via proposal (the 'generator step' of pg-gan)
- Cell 4: run msprime, get a tree sequence
- Cell 5: extract the genotype matrix — **the thing the CNN sees**
- Cell 6: change seed / perturb θ and see how output responds

**Kernel:** Python 3.11 (Mathieson)

## Cell 1 — Setup

We inject the pg-gan-mosquito repo directory onto `sys.path` so we can import their modules as a library without installing them. This is the standard 'use someone else's research code' pattern.

In [ ]:
import sys
from pathlib import Path

# Locate the pg-gan-mosquito repo. Two supported layouts:
#   Option A (recommended for public users): cloned as a sibling of this repo
#   Option B (Preston's local layout): inside Mathieson_Application/
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent

CANDIDATES = [
    REPO_ROOT.parent / "pg-gan-mosquito",
    REPO_ROOT.parent / "Mathieson_Application" / "pg-gan-mosquito",
]
PGGAN_REPO = next((p for p in CANDIDATES if p.exists()), None)

if PGGAN_REPO is None:
    raise FileNotFoundError(
        "pg-gan-mosquito not found. Clone it as a sibling of this repo:\n"
        "  git clone https://github.com/mathiesonlab/pg-gan-mosquito.git"
    )

sys.path.insert(0, str(PGGAN_REPO))

import numpy as np
import matplotlib.pyplot as plt

import param_set
import simulation
import global_vars

print(f"Loaded pg-gan-mosquito modules from: {PGGAN_REPO}")
print(f"NUM_SNPS per region: {global_vars.NUM_SNPS}")
print(f"Region length L: {global_vars.L} bp")
print(f"BATCH_SIZE: {global_vars.BATCH_SIZE}")

## Cell 2 — θ as a concrete Python object

`ParamSet(simulation.dadi_joint_mig)` constructs the parameter vector for the paper's actual 9-parameter model — the one behind Table 1. Default values are pulled straight from the 2017 Ag1000G paper's Supplementary Table 2 (line 72) for the BFA vs GNB population pair.

Each parameter is a `Parameter` object with `.value`, `.min`, `.max`, `.name`. That's what θ actually looks like in code.

### What each parameter means

**Demographic parameters (9) — the ones pg-gan is actually inferring:**

Read the timeline diagram below alongside this table. Everything is past-to-present.

| Symbol | Meaning | Units | Notes |
|---|---|---|---|
| `NI` | **Initial** ancestral population size (deep past) | # individuals | The single ancestor population's N_e way back before anything changed |
| `TG` | Time at which the ancestral population **starts changing size** | years ago | Deep past, before the split |
| `NF` | **Final** ancestral population size, right before the split | # individuals | Ancestor grew (or shrank) exponentially from `NI` → `NF` over `TG` → `TS` |
| `TS` | Time of the **population split** | years ago | When the ancestor became POP1 + POP2 |
| `NI1` | **Initial** size of POP1, immediately after the split | # individuals | POP1's starting N_e right when it became its own population |
| `NF1` | **Final** size of POP1 (present day) | # individuals | POP1's current N_e; POP1 grew exponentially from `NI1` → `NF1` over `TS` → now |
| `NI2` | **Initial** size of POP2, immediately after the split | # individuals | Same story for POP2 |
| `NF2` | **Final** size of POP2 (present day) | # individuals | Same story for POP2 |
| `MG` | **Bidirectional** migration rate between POP1 and POP2 after the split | scaled: `MG = 2·NF·m` | Where `m` is the fractional migration rate per generation. Continuous, not a pulse. Symmetric — same rate in both directions |

**Calibration parameters (2) — species constants, held fixed at biological values:**

| Symbol | Meaning | Units | Notes |
|---|---|---|---|
| `mut` | Point-mutation rate | per site per generation | Mosquito: 3.5×10⁻⁹ (from Drosophila studies, per the 2017 paper) |
| `reco` | Recombination rate | per site per generation | Mosquito: 1.45×10⁻⁸ |

### The timeline

```
       PAST                                                            PRESENT
        ◄─────────────────────────────────────────────────────────────►

           TG                                     TS                       NOW
           │                                      │                        │
           │                                      │                        │
[anc. pop  │                                      │                        │
  size NI ]│──── exponential change ────►[NF]     │                        │
           │      (NI → NF over TG-TS gens)       │                        │
           │                                      │                        │
           │                                      │ SPLIT                  │
           │                                      │                        │
           │                                      │ ┌─POP1: NI1─exp─►[NF1]─┤
           │                                      │ │                      │
           │                                      │ │  ◄── continuous ──►  │
           │                                      │ │      migration MG    │
           │                                      │ │                      │
           │                                      │ └─POP2: NI2─exp─►[NF2]─┤
```

### Two implementation quirks worth noting

**1. Time units in the code are YEARS, not generations.** The `dadi_joint_mig` function internally converts by multiplying by `gen_per_year = 11` before handing values to msprime. So the raw `.value` you'll see printed (e.g. `TS = 4459`) is *4,459 years*, which becomes 49,049 generations inside msprime. This is a Ag1000G-inherited convention — the 2017 paper reported times in years. The paper's Table 1 in the *pg-gan-mosquito* paper reports the same numbers back in generations — mildly confusing but not incorrect.

**2. The migration rate `MG` is scaled.** `MG = 2·NF·m` where `m` is the actual fractional migration rate per generation. Baseline value from the 2017 paper is capped at 20; pg-gan-mosquito raises the cap to 100 to allow the search room to move. Table 1 in the paper reports `MG = 27` for pg-gan-mosquito mig, meaning `m` ≈ 27 / (2·NF).

In [ ]:
params = param_set.ParamSet(simulation.dadi_joint_mig)

LABELS = {
    "NI":   "Number initial (ancestral)",
    "TG":   "Time growth",
    "NF":   "Number final (ancestral)",
    "TS":   "Time split",
    "NI1":  "Number initial pop 1",
    "NF1":  "Number final pop 1",
    "NI2":  "Number initial pop 2",
    "NF2":  "Number final pop 2",
    "MG":   "Migration rate",
    "mut":  "Mutation rate",
    "reco": "Recombination rate",
}

print("θ for the paper's dadi_joint_mig model (BFA vs GN defaults):\n")
print(f"{'name':<6} {'label':<28} {'value':>15} {'min':>15} {'max':>15}")
print("-" * 82)

# 9 demographic + 2 calibration = 11 total
param_names = ["NI", "TG", "NF", "TS", "NI1", "NF1", "NI2", "NF2", "MG", "mut", "reco"]
for name in param_names:
    p = getattr(params, name)
    print(f"{p.name:<6} {LABELS[name]:<28} {p.value:>15.4g} {p.min:>15.4g} {p.max:>15.4g}")

print("\nCompare these initial values to Table 1's 'baseline mig' column in the paper.")
print("The paper's Table 1 'pg-gan-mosquito mig' column is what these become AFTER training.")

## Cell 3 — Perturb each parameter to see what a "θ update" looks like

**What this cell shows:** each demographic parameter (NI, TS, NF1, MG) getting a small random nudge — a "proposed" new value near the current one.

This is exactly the "generator update step" in pg-gan. Every training iteration, the code perturbs the current θ vector to generate several candidate new θ vectors, simulates data under each, and keeps whichever one produces data the discriminator scores as most "real."

**The mechanics:** `p.proposal(current_value, multiplier)` draws a new value from a Normal distribution centered on `current_value`, with width scaled by `multiplier`. The result is clipped to stay inside the parameter's `[min, max]` bounds.

- `multiplier = 1.0` → typical proposal step (default)
- `multiplier = 3.0` → 3× bigger typical jump

pg-gan actually **anneals** the multiplier over training — big jumps early (explore θ space), small jumps late (fine-tune).

In [ ]:
np.random.seed(0)  # deterministic for demo

# For each demographic parameter, show its current value and two candidate
# perturbations (one small, one 3x larger). Each perturbation is what pg-gan
# would call a "proposed" new θ value.

print(f"{'param':<6} {'current value':>16} {'small perturbation':>22} {'big perturbation':>22}")
print(f"{'':<6} {'':<16} {'(multiplier=1.0)':>22} {'(multiplier=3.0)':>22}")
print("-" * 70)

for name in ["NI", "TS", "NF1", "MG"]:
    p = getattr(params, name)
    current = p.value
    small_perturbation = p.proposal(current, multiplier=1.0)
    big_perturbation   = p.proposal(current, multiplier=3.0)
    print(f"{name:<6} {current:>16.4g} {small_perturbation:>22.4g} {big_perturbation:>22.4g}")

print("\nEach cell in this table is one candidate θ value pg-gan might try.")
print("In real training, the discriminator scores each candidate, and the one")
print("that produces the most 'real'-looking simulated data wins and becomes")
print("the new θ.")

## Cell 4 — Run msprime with this θ

This is the 'generator producing fake data' step. `simulation.dadi_joint_mig(params, sample_sizes, seed, reco)` calls msprime under the hood and returns a `TreeSequence` object — msprime's data structure representing the ancestry of the sample.

For the paper's GN+BF setup: 31 GN + 81 BF diploid mosquitoes → 62 + 162 = 224 haplotypes total.

Note: `dadi_joint_mig` uses the **modern msprime 1.x API** (`Demography` + `sim_ancestry` + `sim_mutations`), so no deprecation warnings — unlike the older `im` model which uses the legacy `msprime.simulate` API.

In [ ]:
sample_sizes = [62, 162]   # haplotypes for POP1 and POP2
reco_rate = 1.45e-8         # mosquito recombination rate from the paper

ts = simulation.dadi_joint_mig(params, sample_sizes=sample_sizes, seed=42, reco=reco_rate)

print(f"msprime returned a TreeSequence:")
print(f"  num_samples (haplotypes):   {ts.num_samples}")
print(f"  num_mutations (SNPs):        {ts.num_mutations}")
print(f"  num_sites:                   {ts.num_sites}")
print(f"  sequence_length (bp):        {ts.sequence_length:.0f}")
print(f"  num_trees (recombination):   {ts.num_trees}")

print(f"\nHaplotype breakdown by population:")
for pop_id in range(2):
    samples_in_pop = ts.samples(population=pop_id)
    print(f"  pop {pop_id}: {len(samples_in_pop)} haplotypes")

print(f"\nNote: ~600 SNPs in a 5kb region — mosquito-scale N_e (millions) produces")
print(f"MUCH more variation per region than human-scale N_e (~10k) would.")

### Cell 4b — Turn the `TreeSequence` into a per-SNP DataFrame

The `ts` object contains a lot of structured information (positions, ancestral vs derived alleles, which haplotype carries what). Let's pull it into a pandas DataFrame so you can actually *see* it.

One row per SNP. Columns:
- **`position_bp`** — where the mutation sits along the 5000 bp region
- **`count_POP1`** — how many of the 62 POP1 haplotypes carry the derived allele
- **`count_POP2`** — how many of the 162 POP2 haplotypes carry the derived allele
- **`freq_POP1`, `freq_POP2`** — same but as fractions
- **`abs_freq_diff`** — |freq_POP1 − freq_POP2|; big values mean the SNP is differentiated between populations (F_ST signal)

In [ ]:
import pandas as pd

# SNP positions along the 5 kb region
positions = np.array([site.position for site in ts.sites()])

# Genotype matrix: rows = SNPs, cols = haplotypes
X = ts.genotype_matrix()

# Split haplotypes by population
pop1_haps = list(ts.samples(population=0))   # 62 haplotypes
pop2_haps = list(ts.samples(population=1))   # 162 haplotypes

# Build the DataFrame — one row per SNP
snp_df = pd.DataFrame({
    "position_bp":   positions,
    "count_POP1":    X[:, pop1_haps].sum(axis=1),
    "count_POP2":    X[:, pop2_haps].sum(axis=1),
    "freq_POP1":     X[:, pop1_haps].mean(axis=1).round(3),
    "freq_POP2":     X[:, pop2_haps].mean(axis=1).round(3),
})
snp_df["abs_freq_diff"] = (snp_df["freq_POP1"] - snp_df["freq_POP2"]).abs().round(3)

print(f"DataFrame shape: {snp_df.shape[0]} SNPs × {snp_df.shape[1]} columns\n")
print("First 10 SNPs:")
snp_df.head(10)

## Cell 5 — Extract the genotype matrix — **THE tensor the CNN sees**

`ts.genotype_matrix()` gives you a numpy array of shape `(num_snps, num_haplotypes)` filled with 0s (ancestral allele) and 1s (derived allele).

This is the 'image' from our earlier discussion. Rows = SNPs down the chromosome; columns = haplotypes across individuals. The CNN treats this as a black-and-white image.

In [ ]:
X = ts.genotype_matrix()

print(f"Shape: {X.shape}   (SNPs × haplotypes)")
print(f"dtype: {X.dtype}")
print(f"Values are {set(np.unique(X))}")
print(f"\nMean derived-allele frequency: {X.mean():.3f}")
print(f"Total 1s in matrix: {int(X.sum())}")

print(f"\nFirst 8 SNPs × 20 haplotypes (rows = SNPs, cols = individuals):")
print(X[:8, :20])

### Alternative view: real base-pair positions along the full 5000 bp region

The plot above uses **SNP index** on the y-axis and only shows the first 100 SNPs. Below is the same data plotted with **actual bp positions** on the x-axis — you'll see all 666 SNPs spread across the full region, and where they cluster or gap.

This isn't what the CNN discriminator sees (it uses SNP index + a separate distance channel), but it's better for building intuition about the physical layout along the chromosome.

In [ ]:
# Plot every SNP at its true bp position along the 5000 bp region
# X-axis = base pair position (0 to 5000)
# Y-axis = haplotype index (0-61: POP1, 62-223: POP2)
# Each dot = one haplotype carrying the derived allele at that bp position

positions = np.array([site.position for site in ts.sites()])

# Find all (SNP, haplotype) pairs where value is 1 (derived allele present)
snp_indices, hap_indices = np.where(X == 1)
bp_positions = positions[snp_indices]

fig, ax = plt.subplots(figsize=(14, 6))
ax.scatter(bp_positions, hap_indices, c="black", s=1, marker="s", alpha=0.5)
ax.axhline(y=62, color="red", linestyle="--", linewidth=1, alpha=0.7)
ax.set_xlim(0, 5000)
ax.set_ylim(224, 0)   # flip so haplotype 0 is at the top, like the previous plot
ax.set_xlabel("Base pair position along the 5000 bp region")
ax.set_ylabel("Haplotype index (0-61: POP1 above red, 62-223: POP2 below)")
ax.set_title(f"All {len(positions)} SNPs at true bp positions\n"
             f"Each dot = one haplotype carrying the derived allele at that position")
plt.tight_layout()
plt.show()

print(f"Region: 5000 bp")
print(f"SNPs: {len(positions)}")
print(f"Average spacing: {5000/len(positions):.1f} bp between SNPs")
print(f"\nVertical stripes = a SNP with high derived-allele frequency (many haplotypes carry it).")
print(f"Gaps along the x-axis = stretches of DNA with no variation.")

### Visualize the genotype matrix

Same tensor as above, plotted as a heatmap. Black = ancestral (0), white = derived (1). You're literally looking at the CNN's input.

In [ ]:
# Take a slice for visualization (first 100 SNPs, all haplotypes)
slice_snps = min(100, X.shape[0])
X_slice = X[:slice_snps, :]

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(X_slice, aspect="auto", cmap="binary", interpolation="nearest")
ax.axvline(x=62, color="red", linestyle="--", linewidth=1, alpha=0.7)
ax.set_xlabel("Haplotype index (0-61: POP1, 62-223: POP2)")
ax.set_ylabel("SNP index (going down the chromosome)")
ax.set_title(f"Genotype matrix — first {slice_snps} SNPs × {X.shape[1]} haplotypes\n"
             f"Black = ancestral (0), white = derived (1)")
plt.tight_layout()
plt.show()

print("Red dashed line separates the two populations.")
print("With continuous bidirectional migration (MG=20 baseline) between geographically")
print("close populations, they should look pretty similar — subtle differentiation.")

## Cell 6 — How does the output change with seed vs θ?

Two sources of variation in the simulated data:
1. **Random seed** — stochasticity in the coalescent process. Same θ, different runs, different-looking matrices.
2. **θ values** — the actual demographic scenario. Different N, TS, MG produce systematically different patterns.

The CNN discriminator needs to look at statistical properties of many regions to distinguish (2) from (1).

In [ ]:
# Fix θ, vary seed
print("=== Same θ, different seeds ===\n")
for seed in [1, 42, 99, 2024]:
    ts_s = simulation.dadi_joint_mig(params, sample_sizes=sample_sizes, seed=seed, reco=reco_rate)
    print(f"  seed={seed:>5}: {ts_s.num_mutations:>4} SNPs, {ts_s.num_trees:>4} trees")

# Now perturb θ significantly — try shorter and longer split times
print("\n=== Perturb TS (population split time, in years) ===\n")
original_TS = params.TS.value
for TS_years in [1500, 4459, 10000, 30000]:
    params.TS.value = TS_years
    try:
        ts_t = simulation.dadi_joint_mig(params, sample_sizes=sample_sizes, seed=42, reco=reco_rate)
        print(f"  TS={TS_years:>6} yr: {ts_t.num_mutations:>4} SNPs")
    except ValueError as e:
        print(f"  TS={TS_years:>6} yr: rejected — {e.args[0]}")
params.TS.value = original_TS  # restore

print("\nSNP count varies with seed (noise) and with θ (signal).")
print("The CNN's job is to tell one class of θ apart from another using")
print("MANY regions like this at once.")
print("\nNote: TS must stay less than TG (deep-past ancestral change is older than the split).")

## What you now know

- **θ is a `ParamSet` object** — a bag of `Parameter` instances, each holding `.value`, `.min`, `.max`.
- The paper's actual model, `dadi_joint_mig`, has **9 demographic parameters + 2 calibration constants**: NI, TG, NF, TS, NI1, NF1, NI2, NF2, MG (demographic), mut, reco (calibration).
- **Times are stored in years** in the params but converted to generations internally (×11 for mosquitoes). `TS` must be less than `TG` — the split is more recent than the ancestral size change.
- **`Parameter.proposal()`** is the θ-update step in pg-gan. Normal perturbation, bounded by [min, max].
- **`simulation.dadi_joint_mig(...)`** takes θ and produces an msprime `TreeSequence`.
- **`ts.genotype_matrix()`** returns a `(SNPs × haplotypes)` numpy array of 0/1 — the exact tensor the CNN discriminator scores.
- **Same θ + different seeds → different data** (noise). **Different θ + same seed → systematically different data** (signal). The CNN's job is to tell one class of θ apart from another by looking at statistical properties across many regions.

Next: read `discriminator.py` — the CNN that consumes these matrices.